In [ ]:
%pip install -q langchain langchain-aws langsmith langgraph pydantic boto3 python-dotenv

# Week 5 Friday — Evaluation Harness Engineering

You can build agents. You can observe them with LangSmith. You can give them long-term memory and connect them to knowledge bases with Pinecone and RAG. But here is the question nobody wants to ask out loud: **how do you actually know any of it works correctly?**

You tested it manually a few times. It looked good. You shipped it. Two weeks later you changed a prompt and silently broke three capabilities nobody noticed until a customer complained. This is the story of every AI team that skips evaluation.

**Evaluation harnesses** are the engineering answer to this problem. A harness is a repeatable, automated system that tests your agent against a curated set of inputs, scores its outputs with programmatic evaluators, and produces a report that tells you exactly what improved, what degraded, and what broke. They are to LLM agents what unit tests are to traditional software — except the outputs are non-deterministic, so the evaluators have to be smarter.

Today we build the full stack: datasets, evaluators, pipelines, regression tests, and LangSmith integration.

## Learning Objectives

By the end of today, you will be able to:

- Design evaluation datasets with diverse test cases covering happy paths, edge cases, and adversarial inputs
- Write programmatic evaluators for correctness, relevance, and format compliance
- Build batch evaluation pipelines that score agent outputs at scale
- Implement regression testing to catch prompt and model regressions before they reach production
- Integrate evaluation workflows with LangSmith for tracking and comparison

## Today's Roadmap

| Stage | Topic | Duration |
|-------|-------|----------|
| 1 | Why Evaluation Harnesses Matter | ~15 min |
| 2 | Building Evaluation Datasets | ~25 min |
| 3 | Programmatic Evaluators | ~30 min |
| 4 | Batch Evaluation Pipeline | ~30 min |
| 5 | Regression Testing & LangSmith Integration | ~30 min |

---

# Stage 1 — Why Evaluation Harnesses Matter

### The Problem with Manual Testing

Every developer's first instinct is to test their agent by hand: type a question, read the answer, decide if it looks right. This works fine when you have one prompt and five minutes. It falls apart in every other scenario:

- **It doesn't scale.** You have 50 test cases? You are not running them by hand every time you change a prompt word.
- **It misses edge cases.** You always test the happy path. Nobody manually types "What is 0 divided by 0?" at 3 AM during a deploy.
- **It breaks silently.** You change the system prompt, the model version, or a tool implementation. Nothing crashes. But the agent now gives wrong answers to 15% of queries you never check.
- **It is not comparable.** "I think v2 is better than v1" is not engineering. You need numbers, not vibes.

Manual testing is fine for exploration. It is not a quality strategy.

### What Is an Evaluation Harness?

An evaluation harness has four components:

1. **Dataset** — A curated collection of test cases, each with an input and an expected output (or at minimum, criteria for what a good output looks like). This is your ground truth.

2. **Evaluators** — Functions that score agent outputs. Some are simple (does the output contain the right number?), some are sophisticated (ask another LLM to judge quality on a rubric).

3. **Pipeline** — The orchestration layer that runs the agent on every test case, passes each output through every evaluator, and collects the results.

4. **Reporting** — Aggregation and visualization of scores. Pass rates, score distributions, per-category breakdowns, regression diffs.

The rest of today is building each of these from scratch, then wiring them together into a system you can run with a single function call.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the numeric result."""
    try:
        allowed = {'abs': abs, 'round': round, 'min': min, 'max': max, 'pow': pow}
        result = eval(expression, {'__builtins__': {}}, allowed)
        return str(result)
    except Exception as e:
        return f'Error: {e}'


@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a numeric value between units. Supports miles/km, fahrenheit/celsius, pounds/kg."""
    conversions = {
        ('miles', 'km'): lambda v: v * 1.60934,
        ('km', 'miles'): lambda v: v * 0.621371,
        ('fahrenheit', 'celsius'): lambda v: (v - 32) * 5 / 9,
        ('celsius', 'fahrenheit'): lambda v: v * 9 / 5 + 32,
        ('pounds', 'kg'): lambda v: v * 0.453592,
        ('kg', 'pounds'): lambda v: v * 2.20462,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        return f'Conversion from {from_unit} to {to_unit} is not supported.'
    result = conversions[key](value)
    return f'{value} {from_unit} = {result:.4f} {to_unit}'


model = init_chat_model('bedrock:us.amazon.nova-2-lite-v1:0')
agent = create_react_agent(model, tools=[calculator, unit_converter])

response = agent.invoke({'messages': [{'role': 'user', 'content': 'What is 15% of 240?'}]})
print('Agent response:', response['messages'][-1].content)

response2 = agent.invoke({'messages': [{'role': 'user', 'content': 'Convert 72 fahrenheit to celsius'}]})
print('Agent response:', response2['messages'][-1].content)

print()
print('Manual testing works for two questions.')
print('Can you do this for 50 test cases every time you change a prompt?')

---

# Stage 2 — Building Evaluation Datasets

A harness is only as good as its dataset. If your tests only cover the easy cases, your harness will give you a false sense of confidence. Good evaluation datasets are **diverse**, **representative**, and **tagged** so you can analyze performance by category.

### Anatomy of a Test Case

Every test case in our harness is a dictionary with three fields:

- **`input`** — The exact string you would send to the agent. This is the user's question or instruction.
- **`expected_output`** — What the correct answer looks like. For math, this is the numeric result. For other tasks, it might be keywords, a format, or a reference answer for LLM comparison.
- **`tags`** — A list of labels for categorization. Tags let you answer questions like "how does the agent perform on temperature conversions?" or "are edge cases passing?"

The power of tags cannot be overstated. When your overall pass rate is 80%, tags tell you whether the 20% failure is spread evenly or concentrated in one category you can fix.

In [ ]:
evaluation_dataset = [
    {
        'input': 'What is 15% of 240?',
        'expected_output': '36',
        'tags': ['math', 'percentage', 'easy'],
    },
    {
        'input': 'Calculate 1024 * 768',
        'expected_output': '786432',
        'tags': ['math', 'multiplication', 'easy'],
    },
    {
        'input': 'Convert 100 fahrenheit to celsius',
        'expected_output': '37.7778',
        'tags': ['conversion', 'temperature', 'easy'],
    },
    {
        'input': 'What is the square root of 144?',
        'expected_output': '12',
        'tags': ['math', 'roots', 'easy'],
    },
    {
        'input': 'Convert 26.2 miles to km',
        'expected_output': '42.1647',
        'tags': ['conversion', 'distance', 'medium'],
    },
    {
        'input': 'What is 2**10 + 2**8?',
        'expected_output': '1280',
        'tags': ['math', 'exponents', 'medium'],
    },
    {
        'input': 'If I weigh 180 pounds, how many kg is that?',
        'expected_output': '81.6466',
        'tags': ['conversion', 'weight', 'medium'],
    },
    {
        'input': 'What is (15 + 27) * 3 - 10?',
        'expected_output': '116',
        'tags': ['math', 'order_of_operations', 'medium'],
    },
    {
        'input': 'Convert 0 celsius to fahrenheit',
        'expected_output': '32',
        'tags': ['conversion', 'temperature', 'edge_case'],
    },
    {
        'input': 'What is 0 divided by 5?',
        'expected_output': '0',
        'tags': ['math', 'division', 'edge_case'],
    },
]

print(f'Dataset: {len(evaluation_dataset)} test cases')
print()
for i, tc in enumerate(evaluation_dataset):
    print(f"  [{i+1:2d}] {tc['input'][:50]:50s}  ->  {tc['expected_output']:>10s}  {tc['tags']}")

In [ ]:
from collections import defaultdict

tag_index = defaultdict(list)
for i, tc in enumerate(evaluation_dataset):
    for tag in tc['tags']:
        tag_index[tag].append(i + 1)

print('Dataset organized by tag:\n')
for tag in sorted(tag_index.keys()):
    cases = tag_index[tag]
    print(f'  {tag:25s}  ({len(cases)} cases)  test numbers: {cases}')

difficulty_counts = defaultdict(int)
for tc in evaluation_dataset:
    for tag in tc['tags']:
        if tag in ('easy', 'medium', 'hard', 'edge_case'):
            difficulty_counts[tag] += 1

print('\nDifficulty distribution:')
for level in ['easy', 'medium', 'hard', 'edge_case']:
    count = difficulty_counts.get(level, 0)
    bar = '\u2588' * (count * 3)
    print(f'  {level:12s} {bar} {count}')

capability_counts = defaultdict(int)
for tc in evaluation_dataset:
    for tag in tc['tags']:
        if tag in ('math', 'conversion'):
            capability_counts[tag] += 1

print('\nCapability coverage:')
for cap, count in sorted(capability_counts.items()):
    print(f'  {cap:15s} {count} test cases')

### Where Do Good Test Cases Come From?

Building an evaluation dataset is not a one-time task. It grows over time from multiple sources:

- **Production logs** — Real user queries are the best test cases. If users ask your agent something and it fails, that query belongs in the dataset.
- **Failure reports** — Every bug report is a test case waiting to be written. If someone reports that the agent gave a wrong conversion, encode that exact scenario.
- **LangSmith traces** — Browse your trace history. Sort by latency, look at multi-step chains, find runs where the agent took unexpected paths. These become test cases.
- **Adversarial brainstorming** — Sit down and think about what could go wrong. What if the input is empty? What if the user asks for a conversion you do not support? What if the math expression is malformed?
- **Combinatorial expansion** — You have 3 math operations and 3 conversion types? That is 6 capability categories, each needing easy/medium/hard cases. Build systematically.

A practical target: **start with 20-30 test cases** covering your core capabilities. Add 5 new cases every week from production observations. Within a month you have a serious evaluation suite.

---

# Stage 3 — Programmatic Evaluators

An evaluator is a function that takes agent output (and possibly the input and expected output) and returns a score. Different types of evaluators catch different types of problems:

| Evaluator Type | What It Checks | Return Type | When to Use |
|---------------|----------------|-------------|-------------|
| Exact match | Output equals expected | `bool` | Deterministic answers (math, lookups) |
| Keyword containment | Key terms appear in output | `float` (0-1) | Open-ended responses where certain facts must appear |
| Format validation | Output follows a structure | `bool` | JSON responses, structured outputs, Pydantic models |
| LLM-as-judge | Quality assessed by another LLM | `dict` (score + reasoning) | Subjective quality, tone, helpfulness |
| Semantic similarity | Meaning closeness via embeddings | `float` (0-1) | Paraphrased or reworded correct answers |
| Numeric closeness | Number in output matches expected | `bool` | Math and conversion results with tolerance |

The key insight: **use multiple evaluators per test case.** A math agent might pass the numeric check but fail the format check. An LLM judge might give a high score even when the number is wrong. Each evaluator covers a different failure mode.

In [ ]:
import json
import re
from pydantic import BaseModel, ValidationError
from langchain_aws import BedrockEmbeddings


def exact_match(output: str, expected: str) -> bool:
    return output.strip().lower() == expected.strip().lower()


def contains_keywords(output: str, keywords: list) -> float:
    if not keywords:
        return 0.0
    found = sum(1 for kw in keywords if kw.lower() in output.lower())
    return found / len(keywords)


def format_validator(output: str, pydantic_model: type) -> bool:
    try:
        data = json.loads(output)
        pydantic_model(**data)
        return True
    except (json.JSONDecodeError, ValidationError):
        return False


def llm_judge(input_text: str, output: str, criteria: str) -> dict:
    judge = init_chat_model('bedrock:us.amazon.nova-2-lite-v1:0')
    prompt = (
        'You are an evaluation judge. Score the following AI output on a scale of 1 to 5 '
        'where 1 is terrible and 5 is perfect.\n\n'
        f'Evaluation criteria: {criteria}\n\n'
        f'Original question: {input_text}\n\n'
        f'AI response: {output}\n\n'
        'Respond in exactly this format:\n'
        'SCORE: <number 1-5>\n'
        'REASON: <one sentence explanation>'
    )
    response = judge.invoke(prompt)
    text = response.content
    score_match = re.search(r'SCORE:\s*(\d)', text)
    score = int(score_match.group(1)) if score_match else 3
    score = max(1, min(5, score))
    reason_match = re.search(r'REASON:\s*(.+)', text)
    reasoning = reason_match.group(1).strip() if reason_match else text[:200]
    return {'score': score, 'reasoning': reasoning}


_embed_model = BedrockEmbeddings(model_id='amazon.titan-embed-text-v2:0')

def semantic_similarity(output: str, expected: str, model=None) -> float:
    emb = model or _embed_model
    vectors = emb.embed_documents([output, expected])
    a, b = vectors[0], vectors[1]
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = sum(x ** 2 for x in a) ** 0.5
    mag_b = sum(x ** 2 for x in b) ** 0.5
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)


def numeric_closeness(output: str, expected: str, tolerance: float = 0.01) -> bool:
    numbers_out = re.findall(r'-?\d+\.?\d*', output)
    numbers_exp = re.findall(r'-?\d+\.?\d*', expected)
    if not numbers_out or not numbers_exp:
        return False
    out_val = float(numbers_out[-1])
    exp_val = float(numbers_exp[-1])
    if exp_val == 0:
        return abs(out_val) < tolerance
    return abs(out_val - exp_val) / abs(exp_val) < tolerance


print('Six evaluators defined:')
print()
print('  exact_match(output, expected) -> bool')
print('  contains_keywords(output, keywords) -> float')
print('  format_validator(output, pydantic_model) -> bool')
print('  llm_judge(input_text, output, criteria) -> dict')
print('  semantic_similarity(output, expected, model?) -> float')
print('  numeric_closeness(output, expected, tolerance?) -> bool')

In [ ]:
print('=' * 60)
print('EVALUATOR TEST SUITE')
print('=' * 60)

print('\n--- exact_match ---')
print(f"  '36' vs '36':                  {exact_match('36', '36')}")
print(f"  'Paris' vs 'paris':            {exact_match('Paris', 'paris')}")
print(f"  '36.0' vs '36':                {exact_match('36.0', '36')}")
print(f"  'The answer is 36' vs '36':    {exact_match('The answer is 36', '36')}")

print('\n--- contains_keywords ---')
print(f"  ['36','dollars'] in 'The result is 36 dollars':  {contains_keywords('The result is 36 dollars', ['36', 'dollars'])}")
print(f"  ['36','euros']   in 'The result is 36 dollars':  {contains_keywords('The result is 36 dollars', ['36', 'euros'])}")
print(f"  ['python','code'] in 'Hello world':              {contains_keywords('Hello world', ['python', 'code'])}")

print('\n--- format_validator ---')

class MathResult(BaseModel):
    answer: float
    unit: str

valid_json = '{"answer": 36.0, "unit": "dollars"}'
invalid_text = 'The answer is 36'
missing_field = '{"answer": 36.0}'

print(f'  Valid JSON + valid schema:   {format_validator(valid_json, MathResult)}')
print(f'  Plain text:                  {format_validator(invalid_text, MathResult)}')
print(f'  JSON missing required field: {format_validator(missing_field, MathResult)}')

print('\n--- numeric_closeness ---')
print(f"  '36' vs '36':                          {numeric_closeness('36', '36')}")
print(f"  'The answer is 36.001' vs '36':        {numeric_closeness('The answer is 36.001', '36')}")
print(f"  'The answer is 42' vs '36':            {numeric_closeness('The answer is 42', '36')}")
print(f"  'Result: 37.7778 celsius' vs '37.7778': {numeric_closeness('Result: 37.7778 celsius', '37.7778')}")

print('\n--- semantic_similarity ---')
sim1 = semantic_similarity('36', '36')
sim2 = semantic_similarity('The answer is thirty-six', '36')
sim3 = semantic_similarity('I like pizza', '36')
print(f'  identical strings:         {sim1:.4f}')
print(f'  paraphrased number:        {sim2:.4f}')
print(f'  completely unrelated:       {sim3:.4f}')

print('\n--- llm_judge ---')
judgment = llm_judge(
    'What is 15% of 240?',
    '15% of 240 is 36. I calculated this by multiplying 240 by 0.15.',
    'mathematical correctness and clarity of explanation',
)
print(f'  Score: {judgment["score"]}/5')
print(f'  Reasoning: {judgment["reasoning"]}')

---

# Stage 4 — Batch Evaluation Pipeline

Individual evaluators are useful, but the real power comes from orchestrating them into an automated pipeline. The pattern is straightforward:

1. **For each test case** in the dataset, run the agent on the input
2. **Collect** the raw output text
3. **Pass the output** through every evaluator, collecting scores
4. **Record** scores alongside the test case metadata and tags
5. **Aggregate** all scores into a summary: pass rates, averages, per-tag breakdowns

This is the core of the harness. Once built, you can re-run it any time you change a prompt, swap a model, add a tool, or modify retrieval logic. The pipeline answers one question: **did anything get better or worse?**

In [ ]:
from datetime import datetime


def run_agent_on_input(target_agent, input_text: str) -> str:
    try:
        response = target_agent.invoke({'messages': [{'role': 'user', 'content': input_text}]})
        return response['messages'][-1].content
    except Exception as e:
        return f'AGENT_ERROR: {e}'


def evaluate_agent(target_agent, dataset: list, evaluators: dict = None) -> dict:
    if evaluators is None:
        evaluators = {
            'numeric_match': lambda out, tc: numeric_closeness(out, tc['expected_output']),
            'llm_quality': lambda out, tc: llm_judge(tc['input'], out, 'correctness and helpfulness')['score'] / 5.0,
        }

    results = []
    start_time = datetime.now()

    for i, test_case in enumerate(dataset):
        print(f"  [{i+1}/{len(dataset)}] {test_case['input'][:50]}...")
        agent_output = run_agent_on_input(target_agent, test_case['input'])

        scores = {}
        for eval_name, eval_fn in evaluators.items():
            try:
                scores[eval_name] = eval_fn(agent_output, test_case)
            except Exception as e:
                scores[eval_name] = f'ERROR: {e}'

        results.append({
            'input': test_case['input'],
            'expected': test_case['expected_output'],
            'actual': agent_output,
            'tags': test_case['tags'],
            'scores': scores,
        })

    elapsed = (datetime.now() - start_time).total_seconds()

    pass_count = sum(
        1 for r in results
        if r['scores'].get('numeric_match') is True
    )

    llm_scores = [
        r['scores']['llm_quality'] for r in results
        if isinstance(r['scores'].get('llm_quality'), (int, float))
    ]
    avg_llm = sum(llm_scores) / len(llm_scores) if llm_scores else 0.0

    tag_stats = defaultdict(lambda: {'total': 0, 'passed': 0})
    for r in results:
        passed = r['scores'].get('numeric_match') is True
        for tag in r['tags']:
            tag_stats[tag]['total'] += 1
            if passed:
                tag_stats[tag]['passed'] += 1

    return {
        'results': results,
        'summary': {
            'total_tests': len(results),
            'passed': pass_count,
            'pass_rate': pass_count / len(results) if results else 0,
            'avg_llm_quality': avg_llm,
            'tag_stats': dict(tag_stats),
            'elapsed_seconds': elapsed,
        },
    }


print('evaluate_agent() is ready')
print('Signature: evaluate_agent(target_agent, dataset, evaluators=None) -> dict')

In [ ]:
print('=' * 70)
print('  EVALUATION HARNESS \u2014 FULL RUN')
print('=' * 70)
print()

eval_results = evaluate_agent(agent, evaluation_dataset)

summary = eval_results['summary']

print()
print('=' * 70)
print('  RESULTS REPORT')
print('=' * 70)
print(f"  Total tests:      {summary['total_tests']}")
print(f"  Passed:           {summary['passed']}")
print(f"  Pass rate:        {summary['pass_rate']:.1%}")
print(f"  Avg LLM quality:  {summary['avg_llm_quality']:.2f}")
print(f"  Time elapsed:     {summary['elapsed_seconds']:.1f}s")
print()

print('--- Per-Test Results ---')
for i, r in enumerate(eval_results['results']):
    passed = r['scores'].get('numeric_match') is True
    status = 'PASS' if passed else 'FAIL'
    marker = '  ' if passed else '>>'
    llm = r['scores'].get('llm_quality', 'N/A')
    llm_str = f'{llm:.2f}' if isinstance(llm, float) else str(llm)
    print(f"{marker} [{i+1:2d}] {status}  LLM:{llm_str:>5s}  | {r['input'][:40]:40s}")
    if not passed:
        print(f"             expected: {r['expected']}")
        print(f"             got:      {r['actual'][:60]}")

print()
print('--- Pass Rate by Tag ---')
for tag, stats in sorted(summary['tag_stats'].items()):
    rate = stats['passed'] / stats['total'] if stats['total'] else 0
    bar = '\u2588' * int(rate * 20) + '\u2591' * (20 - int(rate * 20))
    print(f"  {tag:22s} {bar} {stats['passed']}/{stats['total']} ({rate:.0%})")

In [ ]:
def print_score_distribution(results_data: dict):
    results = results_data['results']
    summ = results_data['summary']

    print('PASS / FAIL MAP')
    print('-' * 55)
    for i, r in enumerate(results):
        passed = r['scores'].get('numeric_match') is True
        icon = '\u2713' if passed else '\u2717'
        block = '\u2588\u2588' if passed else '\u2591\u2591'
        print(f"  Test {i+1:2d}  {icon} {block}  {r['input'][:40]}")

    total = summ['total_tests']
    passed_count = summ['passed']
    pct = summ['pass_rate']
    filled = int(pct * 30)
    print(f"\n  {'\u2588' * filled}{'\u2591' * (30 - filled)} {passed_count}/{total} ({pct:.0%})")

    print('\n\nLLM QUALITY SCORE DISTRIBUTION')
    print('-' * 55)
    llm_scores = [
        r['scores'].get('llm_quality', 0)
        for r in results
        if isinstance(r['scores'].get('llm_quality'), (int, float))
    ]
    if llm_scores:
        buckets = {'0.0-0.2': 0, '0.2-0.4': 0, '0.4-0.6': 0, '0.6-0.8': 0, '0.8-1.0': 0}
        for s in llm_scores:
            if s < 0.2:
                buckets['0.0-0.2'] += 1
            elif s < 0.4:
                buckets['0.2-0.4'] += 1
            elif s < 0.6:
                buckets['0.4-0.6'] += 1
            elif s < 0.8:
                buckets['0.6-0.8'] += 1
            else:
                buckets['0.8-1.0'] += 1
        max_count = max(buckets.values()) or 1
        for bucket, count in buckets.items():
            bar_len = int((count / max_count) * 25) if max_count > 0 else 0
            print(f"  {bucket}  {'\u2588' * bar_len:25s} {count}")
        mean_s = sum(llm_scores) / len(llm_scores)
        print(f'\n  Mean: {mean_s:.3f}  |  Min: {min(llm_scores):.3f}  |  Max: {max(llm_scores):.3f}')


print_score_distribution(eval_results)

---

# Stage 5 — Regression Testing & LangSmith Integration

### Regression Testing

You have a working agent and a green evaluation report. Life is good. Then you:

- Change the system prompt to be more concise
- Upgrade from Nova Lite to Nova Pro
- Add a new tool
- Modify the RAG retrieval parameters

Any of these changes could improve some behaviors while degrading others. **Regression testing** means re-running the exact same harness after every change and comparing results to the previous baseline. You want to catch the cases where test #7 used to pass and now it fails — before that regression reaches users.

The workflow:

1. Run the harness and save results as the **baseline**
2. Make a change (prompt, model, tool, config)
3. Run the harness again to get **candidate** results
4. Compare baseline vs candidate: what improved, what degraded, what stayed the same
5. Only deploy the candidate if net quality is positive

In [ ]:
prompt_v1 = (
    'You are a helpful assistant with access to math and conversion tools. '
    'When the user asks a question, use the appropriate tool to calculate the answer. '
    'Show your reasoning and provide the final result clearly.'
)

prompt_v2 = (
    'You are a precise calculation assistant. '
    'Use tools to compute answers. '
    'Respond with ONLY the numeric result. No explanations, no units, just the number.'
)

agent_v1 = create_react_agent(model, tools=[calculator, unit_converter], prompt=prompt_v1)
agent_v2 = create_react_agent(model, tools=[calculator, unit_converter], prompt=prompt_v2)

regression_subset = evaluation_dataset[:6]

print('Running v1 (verbose prompt)...')
results_v1 = evaluate_agent(agent_v1, regression_subset)

print('\nRunning v2 (terse prompt)...')
results_v2 = evaluate_agent(agent_v2, regression_subset)

In [ ]:
print('=' * 70)
print('  REGRESSION REPORT: v1 (verbose) vs v2 (terse)')
print('=' * 70)

improved, degraded, stable_pass, stable_fail = 0, 0, 0, 0

for i, (r1, r2) in enumerate(zip(results_v1['results'], results_v2['results'])):
    v1_pass = r1['scores'].get('numeric_match') is True
    v2_pass = r2['scores'].get('numeric_match') is True
    v1_llm = r1['scores'].get('llm_quality', 0)
    v2_llm = r2['scores'].get('llm_quality', 0)

    if v1_pass and v2_pass:
        label = 'STABLE  \u2713'
        stable_pass += 1
    elif not v1_pass and v2_pass:
        label = 'IMPROVED \u2191'
        improved += 1
    elif v1_pass and not v2_pass:
        label = 'DEGRADED \u2193'
        degraded += 1
    else:
        label = 'BOTH FAIL \u2717'
        stable_fail += 1

    print(f"\n  [{i+1}] {label:14s}  {r1['input'][:42]}")
    v1_llm_str = f'{v1_llm:.2f}' if isinstance(v1_llm, (int, float)) else 'N/A'
    v2_llm_str = f'{v2_llm:.2f}' if isinstance(v2_llm, (int, float)) else 'N/A'
    print(f"       v1: {'PASS' if v1_pass else 'FAIL'}  (LLM quality: {v1_llm_str})")
    print(f"       v2: {'PASS' if v2_pass else 'FAIL'}  (LLM quality: {v2_llm_str})")
    if isinstance(v1_llm, (int, float)) and isinstance(v2_llm, (int, float)):
        delta = v2_llm - v1_llm
        direction = 'better' if delta > 0 else 'worse' if delta < 0 else 'same'
        print(f'       LLM delta: {delta:+.2f} ({direction})')

s1, s2 = results_v1['summary'], results_v2['summary']
print(f"\n{'=' * 70}")
print('  SUMMARY')
print(f"  v1 pass rate: {s1['pass_rate']:.0%}  |  v2 pass rate: {s2['pass_rate']:.0%}")
print(f"  v1 avg LLM:   {s1['avg_llm_quality']:.2f}  |  v2 avg LLM:   {s2['avg_llm_quality']:.2f}")
print(f'\n  Improved: {improved}  |  Degraded: {degraded}  |  Stable pass: {stable_pass}  |  Stable fail: {stable_fail}')

if degraded > 0:
    print(f'\n  \u26a0 REGRESSION DETECTED: {degraded} test(s) degraded. Investigate before deploying v2.')
elif improved > 0:
    print(f'\n  \u2713 v2 shows improvement with no regressions. Safe to deploy.')
else:
    print(f'\n  \u2014 No change in pass/fail outcomes between versions.')

### LangSmith Integration

Everything we have built so far runs locally — the dataset is a Python list, the evaluators are functions, the reports print to the notebook. This is great for development, but in a team environment you need:

- **Shared datasets** that the whole team can access and contribute to
- **Historical results** so you can track quality trends over time
- **Trace-level debugging** to understand why a specific test case failed
- **Comparison views** to diff two evaluation runs side by side

LangSmith provides all of this through its `Client` API. The pattern mirrors what we already built — datasets, evaluators, and runs — but everything is stored in LangSmith's cloud platform where your team can see it.

The key functions:

- `client.create_dataset()` — Create a named dataset in LangSmith
- `client.create_examples()` — Add test cases (inputs + expected outputs) to a dataset
- `client.evaluate()` — Run an agent against a dataset with evaluators, store all results and traces

In [ ]:
from langsmith import Client

ls_client = Client()

dataset_name = 'math-agent-eval-harness'

ls_dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description='Evaluation harness for math and conversion agent - Week 5 Friday',
)
print(f"Created LangSmith dataset: '{ls_dataset.name}'")
print(f'  ID: {ls_dataset.id}')
print(f'  URL: https://smith.langchain.com')
print()

ls_client.create_examples(
    inputs=[{'question': tc['input']} for tc in evaluation_dataset],
    outputs=[{'answer': tc['expected_output']} for tc in evaluation_dataset],
    metadata=[{'tags': tc['tags']} for tc in evaluation_dataset],
    dataset_id=ls_dataset.id,
)
print(f'Added {len(evaluation_dataset)} examples to dataset')
print()


def agent_target(inputs: dict) -> dict:
    response = agent.invoke({'messages': [{'role': 'user', 'content': inputs['question']}]})
    return {'output': response['messages'][-1].content}


def correctness_evaluator(run, example) -> dict:
    predicted = run.outputs.get('output', '')
    expected = example.outputs.get('answer', '')
    passed = numeric_closeness(predicted, expected)
    return {'key': 'correctness', 'score': float(passed)}


experiment_results = ls_client.evaluate(
    agent_target,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix='math-agent-harness',
)

print('LangSmith evaluation complete!')
print('View detailed results, traces, and comparisons at: https://smith.langchain.com')

### Production Evaluation Workflow

In a production setting, evaluation harnesses integrate with your deployment pipeline:

1. **CI/CD Integration** — Run the harness automatically on every pull request that modifies prompts, tools, or model configurations. Block merges if the pass rate drops below a threshold.

2. **Scheduled Runs** — Run the harness daily or weekly against the production model. LLM providers update their models without warning — your harness catches silent regressions.

3. **Alerting** — Set up alerts when pass rates drop below thresholds (e.g., "alert if math accuracy drops below 90%"). Connect to Slack, PagerDuty, or email.

4. **Dataset Growth** — Route a sample of production queries through the harness. When the agent fails on a real query, add it to the dataset automatically.

5. **A/B Evaluation** — Before switching from one prompt version or model to another, run both through the harness. Deploy the one with better scores. This is what we did in the regression test above, but automated.

The harness is not a one-time project. It is a living system that grows more valuable every week as your dataset expands and your evaluators get more sophisticated.

---

# Key Takeaways

### Today's Summary

1. **Manual testing does not scale.** Evaluation harnesses automate quality measurement with datasets, evaluators, pipelines, and reports.

2. **Good datasets are diverse and tagged.** Cover easy, medium, and edge cases. Tag every test case so you can analyze performance by category. Grow the dataset continuously from production logs.

3. **Multiple evaluator types catch different failures.** Exact match, keyword containment, format validation, LLM-as-judge, semantic similarity, and numeric closeness each cover different failure modes. Use several together.

4. **Batch pipelines make evaluation repeatable.** One function call runs your agent against every test case and produces a complete report. No more manual spot-checking.

5. **Regression testing catches silent breakage.** Every time you change a prompt, model, or tool, re-run the harness and compare to the baseline. Deploy only when quality improves or holds steady.

6. **LangSmith provides team-scale evaluation infrastructure.** Shared datasets, stored results, trace-level debugging, and comparison views move evaluation from notebooks into production workflows.

### The Week 5 Journey

| Day | Topic | What You Built |
|-----|-------|----------------|
| Monday | LangSmith Observability | Tracing, debugging, and monitoring agent behavior |
| Tuesday | Pinecone Vector Database | Storing and querying embeddings at scale |
| Wednesday | RAG Pipelines | Connecting agents to knowledge bases for grounded answers |
| Thursday | Advanced RAG Patterns | Chunking strategies, hybrid search, re-ranking |
| **Friday** | **Evaluation Harness Engineering** | **Systematic testing, scoring, regression detection, LangSmith integration** |

You started the week watching your agents work. You end it knowing whether they work *correctly*.

### Looking Ahead — Week 6

Next week introduces **LangGraph for advanced agent orchestration**: multi-agent systems, human-in-the-loop workflows, stateful conversation management, and complex graph-based reasoning chains. The evaluation harness you built today will be essential — as your agents get more sophisticated, systematic testing becomes even more critical.